## 1. Setup & Data Loading

In [ ]:
# !pip install autogluon

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# AutoGluon
from autogluon.tabular import TabularDataset, TabularPredictor

In [ ]:
# Data paths - adjust for your environment
# Local
DATA_PATH = Path("data")

# JupyterHub (uncomment if needed)
# DATA_PATH = Path("/home/jovyan/__DATA/APBDID_F25/data/handm")

In [ ]:
# Load datasets
articles_df = pd.read_csv(DATA_PATH / "articles.csv")
customers_df = pd.read_csv(DATA_PATH / "customers.csv")
transactions_df = pd.read_csv(DATA_PATH / "transactions_train.csv")

print(f"Articles: {articles_df.shape}")
print(f"Customers: {customers_df.shape}")
print(f"Transactions: {transactions_df.shape}")

In [ ]:
# Quick look at transactions (our main table)
transactions_df.head()

## 2. Minimal Preprocessing

Keep it simple:
- Sample transactions (full dataset is too large for quick experiments)
- Merge customer and article features
- Target: `article_id` (what article was purchased)

In [ ]:
# Sample for speed - use more data for better results
SAMPLE_SIZE = 50_000
np.random.seed(42)

transactions_sample = transactions_df.sample(n=SAMPLE_SIZE, random_state=42)
print(f"Sampled transactions: {len(transactions_sample):,}")

In [ ]:
# Select key features from articles (keep it minimal)
articles_features = articles_df[[
    "article_id",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "index_group_name",
    "garment_group_name"
]].copy()

# Select key features from customers
customers_features = customers_df[[
    "customer_id",
    "club_member_status",
    "fashion_news_frequency",
    "age"
]].copy()

# Fill missing values
customers_features["club_member_status"] = customers_features["club_member_status"].fillna("UNKNOWN")
customers_features["fashion_news_frequency"] = customers_features["fashion_news_frequency"].fillna("UNKNOWN")
customers_features["age"] = customers_features["age"].fillna(customers_features["age"].median())

In [ ]:
# Merge all features into one training table
train_df = transactions_sample.merge(customers_features, on="customer_id", how="left")
train_df = train_df.merge(articles_features, on="article_id", how="left")

print(f"Training data shape: {train_df.shape}")
train_df.head()

In [ ]:
# Simplify target: predict product_group_name instead of exact article_id
# (article_id has too many unique values for quick training)
TARGET = "product_group_name"

print(f"Target classes: {train_df[TARGET].nunique()}")
train_df[TARGET].value_counts()

In [ ]:
# Features for training (drop IDs and redundant columns)
drop_cols = ["customer_id", "article_id", "t_dat", "price", "sales_channel_id"]
feature_cols = [c for c in train_df.columns if c not in drop_cols and c != TARGET]

train_data = train_df[feature_cols + [TARGET]].copy()
print(f"Features: {feature_cols}")
print(f"Final training shape: {train_data.shape}")

## 3. Train AutoGluon Model

AutoGluon handles:
- Automatic feature engineering
- Model selection & ensembling
- Hyperparameter tuning

In [ ]:
# Convert to AutoGluon dataset
train_ag = TabularDataset(train_data)

In [ ]:
# Train model with time limit (quick baseline)
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="accuracy",
    path="models/autogluon_baseline"
).fit(
    train_ag,
    time_limit=120,  # 2 minutes for quick demo
    presets="medium_quality"  # Options: best_quality, high_quality, medium_quality, optimize_for_deployment
)

## 4. Evaluate Results

In [ ]:
# Model leaderboard
predictor.leaderboard(silent=True)

In [ ]:
# Feature importance
predictor.feature_importance(train_ag)

In [ ]:
# Quick prediction example
sample_customer = train_data.drop(columns=[TARGET]).iloc[:5]
predictions = predictor.predict(sample_customer)
print("Predictions:")
print(predictions)

## 5. Next Steps

To improve the model:
1. **More data**: Increase `SAMPLE_SIZE`
2. **Better features**: Add lag features from transaction history (as shown in class)
3. **Longer training**: Increase `time_limit` or use `best_quality` preset
4. **Different target**: Try predicting `department_name` or `index_group_name`
5. **Evaluation**: Use proper train/validation/test split by time